# Swin-Base Gradient Attacks on ImageNet-100

Mirrors `Model Training/cnn-attacks.ipynb` — same structure, PyTorch/Swin instead of TF/CNN.

**Attacks:**
- FGSM — Fast Gradient Sign Method (ε ∈ {0.005, 0.01, 0.02, 0.04})
- PGD  — Projected Gradient Descent (10 steps, same ε grid)
- DeepFool — minimal-perturbation iterative attack

**Outputs (`Swin_Training/gradient_results/`):**
- `fgsm_results.json`, `pgd_results.json`, `deepfool_results.json`
- `gradient_full_results.json` (combined)
- `confusion_analysis/` — confusion matrices, SSS scores, top confused pairs
- `figures/` — accuracy-vs-epsilon plots, adversarial examples
- `gradient_results_table.tex` — LaTeX-ready table

In [ ]:
import sys, os, json, logging, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import SwinForImageClassification, AutoImageProcessor
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

NOTEBOOK_DIR = Path('.').resolve()              # VIT/
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR / 'src'))

from swin_utils import (
    seed_everything, get_device, ImageNet100Dataset,
    fgsm_attack_swin, pgd_attack_swin, deepfool_attack_swin,
    evaluate_clean, evaluate_under_attack,
    build_adversarial_confusion_swin, compute_sss_from_confusion,
    normalize_imagenet, denormalize_imagenet,
)

# ── Logging ──
LOG_DIR = PROJECT_ROOT / 'Swin_Training' / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path  = LOG_DIR / f'swin_gradient_attacks_{timestamp}.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    handlers=[logging.FileHandler(log_path), logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger('swin_attacks')
logger.info(f'Log: {log_path}')

In [ ]:
# ── Configuration ──
SEED        = 42
CHECKPOINT  = 'microsoft/swin-base-patch4-window7-224'
NUM_CLASSES = 100
BATCH_SIZE  = 16   # Smaller than training — attacks need grad, more memory
NUM_WORKERS = 4
PGD_STEPS   = 10
ADV_EPSILONS = [0.005, 0.01, 0.02, 0.04]
TOP_K_CONFUSED = 20

DATA_DIR     = PROJECT_ROOT / 'ImageNet100_Training' / 'data'
MAPPING_FILE = PROJECT_ROOT / 'ImageNet100_Training' / 'class_mapping.json'
CKPT_FILE    = PROJECT_ROOT / 'Swin_Training' / 'checkpoints' / 'swin_base_imagenet100_best.pt'
BASELINES_FILE = PROJECT_ROOT / 'Swin_Training' / 'clean_baselines' / 'swin_base_clean.json'

OUT_DIR      = PROJECT_ROOT / 'Swin_Training' / 'gradient_results'
FIG_DIR      = OUT_DIR / 'figures'
CONF_DIR     = OUT_DIR / 'confusion_analysis'
CONF_FIG_DIR = CONF_DIR / 'figures'

for d in [OUT_DIR, FIG_DIR, CONF_DIR, CONF_FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

seed_everything(SEED)
device = get_device()
logger.info(f'Device: {device}  Epsilons: {ADV_EPSILONS}')

# ── Load class mapping ──
with open(MAPPING_FILE) as f:
    class_mapping = json.load(f)
class_names = [
    class_mapping[s]['human_name']
    for s in sorted(class_mapping, key=lambda x: class_mapping[x]['index'])
]

In [ ]:
# ── Load test DataLoader ──
processor  = AutoImageProcessor.from_pretrained(CHECKPOINT)
test_ds    = ImageNet100Dataset(DATA_DIR, split='test', processor=processor)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
logger.info(f'Test samples: {len(test_ds)}  Batches: {len(test_loader)}')

In [ ]:
# ── Load trained Swin model ──
assert CKPT_FILE.exists(), f'Checkpoint not found: {CKPT_FILE}\nRun swin-training.ipynb first.'

model = SwinForImageClassification.from_pretrained(CHECKPOINT)
in_features      = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)

state = torch.load(CKPT_FILE, map_location=device)
model.load_state_dict(state['model'])
model = model.to(device)
model.eval()

logger.info(f'Loaded checkpoint: {CKPT_FILE.name}  (val_acc={state["val_acc"]:.4f})')

# ── Verify clean accuracy matches saved baseline ──
clean_results = evaluate_clean(model, test_loader, device, num_classes=NUM_CLASSES)
logger.info(f'Test clean accuracy: {clean_results["accuracy"]:.4f}')
logger.info(f'Mean confidence    : {clean_results["mean_confidence"]:.4f}')

In [ ]:
# ── Run FGSM on all epsilons ──
logger.info('\n[1/3] FGSM Attack')
fgsm_results = {}

for eps in ADV_EPSILONS:
    t0 = time.time()
    res = evaluate_under_attack(
        model, test_loader, device,
        attack_fn=fgsm_attack_swin,
        epsilon=eps,
    )
    elapsed = time.time() - t0
    fgsm_results[eps] = res
    logger.info(f'  FGSM ε={eps}: clean={res["clean_accuracy"]:.3f}  '
                f'adv={res["adv_accuracy"]:.3f}  '
                f'fool={res["fooling_rate"]:.3f}  ({elapsed:.1f}s)')

with open(OUT_DIR / 'fgsm_results.json', 'w') as f:
    json.dump({str(k): v for k, v in fgsm_results.items()}, f, indent=2)
logger.info('FGSM results saved.')

In [ ]:
# ── Run PGD on all epsilons ──
logger.info('\n[2/3] PGD Attack')
pgd_results = {}

for eps in ADV_EPSILONS:
    t0 = time.time()
    res = evaluate_under_attack(
        model, test_loader, device,
        attack_fn=pgd_attack_swin,
        epsilon=eps,
        steps=PGD_STEPS,
        step_size=eps / 4.0,
    )
    elapsed = time.time() - t0
    pgd_results[eps] = res
    logger.info(f'  PGD ε={eps}: clean={res["clean_accuracy"]:.3f}  '
                f'adv={res["adv_accuracy"]:.3f}  '
                f'fool={res["fooling_rate"]:.3f}  ({elapsed:.1f}s)')

with open(OUT_DIR / 'pgd_results.json', 'w') as f:
    json.dump({str(k): v for k, v in pgd_results.items()}, f, indent=2)
logger.info('PGD results saved.')

In [ ]:
# ── Run DeepFool ──
# DeepFool is sample-by-sample and slow (~100 class evals/step/sample).
# Run on first 200 test samples for representative statistics.
logger.info('\n[3/3] DeepFool Attack (200 samples)')
DF_SAMPLES   = 200
df_loader = DataLoader(
    torch.utils.data.Subset(test_ds, list(range(DF_SAMPLES))),
    batch_size=1, shuffle=False, num_workers=0
)

all_l2_dists, total, fooled = [], 0, 0
df_correct_clean, df_correct_adv = 0, 0

for pv, labels in df_loader:
    pv = pv.to(device)
    labels_t = labels.to(device) if isinstance(labels, torch.Tensor) else torch.tensor(labels, device=device)

    with torch.no_grad():
        clean_pred = model(pixel_values=pv).logits.argmax(-1)
    df_correct_clean += (clean_pred == labels_t).sum().item()

    adv_pv, l2_dist = deepfool_attack_swin(model, pv, labels_t, num_classes=NUM_CLASSES)
    with torch.no_grad():
        adv_pred = model(pixel_values=adv_pv).logits.argmax(-1)

    df_correct_adv += (adv_pred == labels_t).sum().item()
    fooled += (adv_pred != clean_pred).sum().item()
    all_l2_dists.append(l2_dist.item())
    total += 1

deepfool_results = {
    'num_samples':     total,
    'clean_accuracy':  df_correct_clean / total,
    'adv_accuracy':    df_correct_adv / total,
    'fooling_rate':    fooled / total,
    'mean_l2_distance': float(np.mean(all_l2_dists)),
    'median_l2_distance': float(np.median(all_l2_dists)),
    'std_l2_distance':  float(np.std(all_l2_dists)),
}

with open(OUT_DIR / 'deepfool_results.json', 'w') as f:
    json.dump(deepfool_results, f, indent=2)
logger.info(f'  DeepFool: fool={deepfool_results["fooling_rate"]:.3f}  '
            f'mean_L2={deepfool_results["mean_l2_distance"]:.4f}')

In [ ]:
# ── Results summary table ──
rows = []
for eps in ADV_EPSILONS:
    f = fgsm_results[eps]
    p = pgd_results[eps]
    rows.append({
        'Epsilon': eps,
        'FGSM_clean_acc': round(f['clean_accuracy'], 4),
        'FGSM_adv_acc':   round(f['adv_accuracy'],   4),
        'FGSM_fool_rate': round(f['fooling_rate'],    4),
        'FGSM_clean_conf': round(f['mean_clean_conf'], 4),
        'FGSM_adv_conf':  round(f['mean_adv_conf'],  4),
        'PGD_adv_acc':    round(p['adv_accuracy'],   4),
        'PGD_fool_rate':  round(p['fooling_rate'],   4),
        'PGD_adv_conf':   round(p['mean_adv_conf'],  4),
    })
df_summary = pd.DataFrame(rows)
df_summary.to_csv(OUT_DIR / 'gradient_results_summary.csv', index=False)
print(df_summary.to_string(index=False))
logger.info('Summary table saved.')

In [ ]:
# ── LaTeX table ──
tex = [
    r'\begin{table}[h]',
    r'\centering',
    r'\caption{Swin-Base adversarial accuracy on ImageNet-100}',
    r'\begin{tabular}{lcccccc}',
    r'\toprule',
    r'$\varepsilon$ & \multicolumn{3}{c}{FGSM} & \multicolumn{3}{c}{PGD} \\',
    r'& Adv Acc & Fool Rate & Adv Conf & Adv Acc & Fool Rate & Adv Conf \\',
    r'\midrule',
]
for _, row in df_summary.iterrows():
    tex.append(
        f'{row["Epsilon"]} & {row["FGSM_adv_acc"]:.3f} & {row["FGSM_fool_rate"]:.3f} & '
        f'{row["FGSM_adv_conf"]:.3f} & {row["PGD_adv_acc"]:.3f} & '
        f'{row["PGD_fool_rate"]:.3f} & {row["PGD_adv_conf"]:.3f} \\\\'
    )
tex += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
with open(OUT_DIR / 'gradient_results_table.tex', 'w') as f:
    f.write('\n'.join(tex))
logger.info('LaTeX table saved.')

In [ ]:
# ── Accuracy vs Epsilon plot ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
for ax, attack, results_dict, label in [
    (axes[0], 'FGSM', fgsm_results, 'Adversarial Accuracy'),
    (axes[1], 'PGD',  pgd_results,  'Adversarial Accuracy'),
]:
    eps_vals  = ADV_EPSILONS
    clean_acc = [results_dict[e]['clean_accuracy'] for e in eps_vals]
    adv_acc   = [results_dict[e]['adv_accuracy']   for e in eps_vals]
    fool_rate = [results_dict[e]['fooling_rate']    for e in eps_vals]

    ax.plot(eps_vals, clean_acc, 'o--', color='green',  label='Clean Acc', linewidth=2)
    ax.plot(eps_vals, adv_acc,   'o-',  color='tomato', label='Adv Acc',   linewidth=2)
    ax.plot(eps_vals, fool_rate, 's-',  color='steelblue', label='Fool Rate', linewidth=2)
    ax.set_xlabel('Epsilon (ε)'); ax.set_ylabel('Rate')
    ax.set_title(f'{attack} — Swin-Base on ImageNet-100')
    ax.set_ylim(0, 1); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIG_DIR / 'accuracy_vs_epsilon.png', dpi=150)
plt.show(); plt.close()
logger.info('Accuracy vs epsilon plot saved.')

In [ ]:
# ── Adversarial example visualization ──
model.eval()
vis_batch, vis_labels = next(iter(
    DataLoader(torch.utils.data.Subset(test_ds, list(range(8))),
               batch_size=8, shuffle=False)
))
vis_batch  = vis_batch.to(device)
vis_labels = vis_labels.to(device) if isinstance(vis_labels, torch.Tensor) else torch.tensor(vis_labels, device=device)

eps_vis = 0.02
adv_vis = fgsm_attack_swin(model, vis_batch, vis_labels, eps_vis)

with torch.no_grad():
    clean_preds = model(pixel_values=vis_batch).logits.argmax(-1)
    adv_preds   = model(pixel_values=adv_vis).logits.argmax(-1)

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for i in range(8):
    clean_img = denormalize_imagenet(vis_batch[i:i+1])[0].permute(1,2,0).cpu().numpy()
    adv_img   = denormalize_imagenet(adv_vis[i:i+1])[0].permute(1,2,0).cpu().numpy()
    pert_img  = ((adv_img - clean_img + 1) / 2).clip(0, 1)  # visualize perturbation

    axes[0, i].imshow(clean_img.clip(0, 1))
    axes[0, i].set_title(f'True: {class_names[vis_labels[i].item()][:10]}\n'
                          f'Pred: {class_names[clean_preds[i].item()][:10]}', fontsize=7)
    axes[0, i].axis('off')

    axes[1, i].imshow(adv_img.clip(0, 1))
    axes[1, i].set_title(f'Adv: {class_names[adv_preds[i].item()][:10]}', fontsize=7,
                          color='red' if adv_preds[i] != vis_labels[i] else 'green')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Clean', fontsize=9)
axes[1, 0].set_ylabel(f'FGSM ε={eps_vis}', fontsize=9)
plt.suptitle(f'Adversarial Examples — Swin-Base FGSM ε={eps_vis}')
plt.tight_layout()
fig.savefig(FIG_DIR / f'adversarial_examples_fgsm_eps{eps_vis}.png', dpi=150)
plt.show(); plt.close()

In [ ]:
# ── Confusion direction analysis (SSS) ──
logger.info('\nBuilding confusion matrices and computing SSS...')
sss_data  = {}
all_top_pairs = {}

for attack_name, attack_fn, attack_kwargs in [
    ('fgsm', fgsm_attack_swin, {}),
    ('pgd',  pgd_attack_swin,  {'steps': PGD_STEPS, 'step_size': None}),
]:
    for eps in ADV_EPSILONS:
        t0 = time.time()
        C, records = build_adversarial_confusion_swin(
            model, test_loader, device,
            attack_fn=attack_fn, epsilon=eps,
            num_classes=NUM_CLASSES, **attack_kwargs
        )
        sss = compute_sss_from_confusion(C)
        key = f'SwinBase_{attack_name}_eps{eps}'
        sss_data[key] = sss

        # Top-K confused pairs
        pairs = []
        for ti in range(NUM_CLASSES):
            for aj in range(NUM_CLASSES):
                if ti != aj and C[ti, aj] > 0:
                    pairs.append({'true_class': class_names[ti], 'adv_class': class_names[aj],
                                  'true_label': ti, 'adv_label': aj, 'count': int(C[ti, aj])})
        pairs.sort(key=lambda x: x['count'], reverse=True)
        all_top_pairs[key] = pairs[:TOP_K_CONFUSED]

        # Save confusion matrix CSV
        df_C = pd.DataFrame(C, index=class_names, columns=class_names)
        df_C.to_csv(CONF_DIR / f'confusion_SwinBase_{attack_name}_eps{eps}.csv')

        logger.info(f'  {attack_name.upper()} ε={eps}: SSS={sss:.4f}  '
                    f'top pair: {pairs[0]["true_class"]}→{pairs[0]["adv_class"]} '
                    f'(n={pairs[0]["count"]})  ({time.time()-t0:.1f}s)')

In [ ]:
# ── SSS comparison plot ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for idx, attack_name in enumerate(['fgsm', 'pgd']):
    ax = axes[idx]
    eps_vals = ADV_EPSILONS
    sss_vals = [sss_data.get(f'SwinBase_{attack_name}_eps{e}', 0) for e in eps_vals]
    ax.plot(eps_vals, sss_vals, 'o-', label='SwinBase', color='steelblue', linewidth=2)
    ax.set_xlabel('Epsilon (ε)'); ax.set_ylabel('Semantic Structure Score')
    ax.set_title(f'{attack_name.upper()} SSS — Swin-Base')
    ax.set_ylim(0, 1); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Semantic Structure Score (SSS) under Gradient Attacks')
plt.tight_layout()
fig.savefig(CONF_FIG_DIR / 'sss_comparison_swin.png', dpi=150)
plt.show(); plt.close()

# Save SSS CSV
sss_rows = []
for key, sss_val in sss_data.items():
    parts = key.split('_')
    sss_rows.append({'Model': parts[0], 'Attack': parts[1],
                     'Epsilon': float(parts[2].replace('eps', '')), 'SSS': sss_val})
pd.DataFrame(sss_rows).to_csv(CONF_DIR / 'sss_scores_swin.csv', index=False)

# Save top confused pairs
with open(CONF_DIR / 'top_confused_pairs_swin.json', 'w') as f:
    json.dump(all_top_pairs, f, indent=2)
logger.info('SSS analysis complete.')

In [ ]:
# ── SSS LaTeX table ──
tex = [
    r'\begin{table}[h]',
    r'\centering',
    r'\caption{Semantic Structure Score (SSS) — Swin-Base on ImageNet-100}',
    r'\begin{tabular}{ll' + 'c' * len(ADV_EPSILONS) + '}',
    r'\toprule',
    'Model & Attack & ' + ' & '.join([f'$\\varepsilon={e}$' for e in ADV_EPSILONS]) + r' \\',
    r'\midrule',
]
for attack_name in ['fgsm', 'pgd']:
    cells = [f'{sss_data.get(f"SwinBase_{attack_name}_eps{e}", 0):.3f}' for e in ADV_EPSILONS]
    tex.append(f'SwinBase & {attack_name.upper()} & ' + ' & '.join(cells) + r' \\')
tex += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
with open(CONF_DIR / 'sss_table_swin.tex', 'w') as f:
    f.write('\n'.join(tex))

In [ ]:
# ── Save combined results ──
all_results = {
    'model': 'SwinBase',
    'dataset': 'ImageNet-100',
    'fgsm': {str(k): v for k, v in fgsm_results.items()},
    'pgd':  {str(k): v for k, v in pgd_results.items()},
    'deepfool': deepfool_results,
    'sss': sss_data,
    'timestamp': datetime.now().isoformat(),
}
with open(OUT_DIR / 'gradient_full_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
logger.info(f'Full results saved: {OUT_DIR / "gradient_full_results.json"}')

In [ ]:
# ── Summary ──
print('=' * 60)
print('  SWIN-BASE GRADIENT ATTACKS SUMMARY')
print('=' * 60)
print(f'  Clean accuracy: {clean_results["accuracy"]:.4f}')
print(f'\n  Adversarial accuracy (ε={ADV_EPSILONS[-1]}):')
print(f'    FGSM: {fgsm_results[ADV_EPSILONS[-1]]["adv_accuracy"]:.4f}')
print(f'    PGD:  {pgd_results[ADV_EPSILONS[-1]]["adv_accuracy"]:.4f}')
print(f'  DeepFool fooling rate: {deepfool_results["fooling_rate"]:.4f}  '
      f'mean L2={deepfool_results["mean_l2_distance"]:.4f}')
print(f'\n  SSS (ε={ADV_EPSILONS[-1]}):')
for atk in ['fgsm', 'pgd']:
    key = f'SwinBase_{atk}_eps{ADV_EPSILONS[-1]}'
    print(f'    {atk.upper()}: {sss_data.get(key, 0):.4f}')
print(f'\n  Top confused pair (FGSM, ε={ADV_EPSILONS[-1]}):')
top_key = f'SwinBase_fgsm_eps{ADV_EPSILONS[-1]}'
if all_top_pairs.get(top_key):
    p = all_top_pairs[top_key][0]
    print(f'    {p["true_class"]} → {p["adv_class"]} (n={p["count"]})')
print(f'\n  Log: {log_path}')
print('=' * 60)